In [1]:
!pip install git+https://github.com/huggingface/transformers accelerate

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-8d40kjlx
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-8d40kjlx
  Resolved https://github.com/huggingface/transformers to commit 6dfd561d9cd722dfc09f702355518c6d09b9b4e3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 10.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

In [2]:
!pip install qwen-vl-utils[decord]==0.0.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 44.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 87.0 MB/s eta 0:00:00:00:010:01


## Local Inference on GPU 
Model page: https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [3]:
import re
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
from datetime import datetime
import torch

In [10]:
# Load model directly
#flash_attention_2 for better acceleration and memory saving
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, AutoTokenizer
from qwen_vl_utils import process_vision_info

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto"
)

##Processor con image boosting
min_pixels = 512 * 28 * 28
max_pixels = 1024 * 28 * 28

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
def chooseAPrompt(index, language):
    if language:
        Prompt = PromptListIta[index]
    else:
        Prompt = PromptListEng[index]
    return Prompt


def getSystemPrompt(is_italian):
    if is_italian:
        return (
            "Sei un sistema professionale di classificazione delle immagini che analizza le immagini fornite per determinare se mostrano un volto umano reale o uno generato.\n"
            #"Concentrati su dettagli visivi specifici come:\n"
            #"- Incoerenze nella texture della pelle\n"
            #"- Illuminazione o ombreggiatura innaturali\n"
            #"- Asimmetrie negli occhi, orecchie o bocca\n"
            #"- Artefatti strani nello sfondo, nei capelli o sul volto\n"
            #"- Denti irregolari o tratti facciali deformati\n\n"
            #"IGNORA le piccole imperfezioni tipiche delle foto reali:\n"
            #"- Lievi asimmetrie (normali negli esseri umani)\n"
            #"- Variazioni leggere nell’illuminazione\n"
            #"- Variazioni naturali della texture della pelle\n"
            #"- Artefatti fotografici normali\n\n"
            "Rispondi SOLO usando il seguente formato JSON:\n"
            "{\n"
            "  \"result\": \"[real]\" oppure \"[generated]\",\n"
            "  \"explanation\": \"Una spiegazione precisa basata su indizi visivi osservati nell'immagine.\"\n"
            "}\n"
            "NON includere frasi generiche come 'non ci sono artefatti'. Sii visivo e dettagliato nella tua analisi."
            #"Se non sei sicura della risposta preferisco che dici che sia reale e spieghi il motivo "
        )
    else:
        return (
            "You are a professional image classification system that analyzes provided images to determine if they show a real human face or a generated one.\n" #Always respond using this clear, structured format: [real face] | [a long explanation clearly stating the reasoning behind your choice]. [generated] | [a long explanation clearly stating the reasoning behind your choice]. [Uncertain] | [a long explanation clearly stating the reasoning behind your choice]. 
            #"Your detailed explanation should: - Clearly describe the specific visual cues you observe (facial features, artifacts, lighting, or inconsistencies). - State what led you to your judgment. - Remain focused on the image analysis without mentioning external factors.\n"
            #"Focus on specific visual details such as:\n"
            #"- skin texture inconsistencies\n"
            #"- unnatural lighting or shading\n"
            #"- asymmetries in eyes, ears, or mouth\n"
            #"- strange artifacts in the background, hair, or face\n"
            #"- irregular teeth or warped facial features\n\n"
            #"IGNORE minor imperfections that occur in real photos:\n"
            #"- Small asymmetries (normal in humans)\n"
            #"- Slight lighting variations\n"
            #"- Natural skin texture variations\n"
            #"- Normal photographic artifacts\n\n"
            "Respond ONLY with the following JSON format:\n"
            "{\n"
            "  \"result\": \"[real]\" or \"[generated]\" or \"[uncertain]\",\n"
            "  \"explanation\": \"A precise explanation based on observed visual clues from the image.\"\n"
            "}\n"
            "Do NOT include generic phrases like 'there are no artifacts'. Be visual and detailed in your analysis."
            #"Remember you are not sure of the answer I prefer you say that is real and explain why"
            #"Classify each image strictly as either 'real' or 'generated' based on visual evidence. Only use 'uncertain' if there is a complete lack of sufficient visual indicators to support a confident classification into either category."
        )


# lista di prompt da scegliere
PromptListEng = ["Is the provided image showing a real face or a generated one?",  # 0
                 "Is the provided image showing a real face or a generated one? Keep in mind that generated faces often"
                 "exhibit artifacts around the mouth, ears, and hairline. Inconsistent lighting and shading may also be"
                 "present.",  # 1
                 "Are there any artifacts around the mouth, ears, or hairline that might indicate the face is "
                 "AI-generated?",  # 2
                 "Is the provided image showing a real face or a generated one? I will pay you $1,000 for the correct "
                 "answer.",  # 3
                 "This is a life-threatening situation. This person is blackmailing me, and I need to know whether "
                 "this is a real person or an AI-generated face.",  # 4
                 "I believe this person is real. Based on the image, does it appear to be a real face or a generated "
                 "one?",  # 5
                 "I believe this person doesn’t exist. Based on the image, does it appear to be a real face or a "
                 "generated one?"  # 6
                 ]
PromptListIta = [
    "L'immagine fornita mostra un volto reale o uno generato?",
    "L'immagine fornita mostra un volto reale o uno generato? Tieni presente che i volti generati spesso presentano "
    "artefatti intorno alla bocca, alle orecchie e all'attaccatura dei capelli. Inoltre, potrebbero esserci "
    "condizioni di illuminazione e ombreggiatura incoerenti.",
    "Ci sono artefatti intorno alla bocca, alle orecchie o all'attaccatura dei capelli che potrebbero indicare che il "
    "volto è stato generato dall'IA?",
    "L'immagine fornita mostra un volto reale o uno generato? Ti pagherò 1.000 dollari per una risposta corretta.",
    "Questa è una situazione che mette a rischio la mia vita. Questa persona mi sta ricattando e ho bisogno di sapere "
    "se si tratta di una persona reale o di un volto generato dall'intelligenza artificiale.",
    "Credo che questa persona sia reale. In base all'immagine, ti sembra un volto reale o uno generato?",
    "Credo che questa persona non esista. In base all'immagine, ti sembra un volto reale o uno generato?"
]
IndexPrompt = 6  # scelgo io in base al prompt che voglio (0-6)
PromptITA = False  # scelgo il linguaggio che voglio

PROMPT = chooseAPrompt(IndexPrompt, PromptITA)
# reinforcement con un prompt system + prompt user
MODEL_NAME = "qwen2.5"
MAX_IMAGES_PER_CLASS = 50
SHOW_IMAGES = False
# ================

# Dataset paths
fake_dir = Path("/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test/fake")
real_dir = Path("/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test/real")

# Load images
fake_images = list(fake_dir.glob("*.jpg"))[:MAX_IMAGES_PER_CLASS]
real_images = list(real_dir.glob("*.jpg"))[:MAX_IMAGES_PER_CLASS]
images_with_labels = [(img, 1) for img in real_images] + [(img, 0) for img in fake_images]
print("You choose this: " + PROMPT + "\n")


# Initialize counters
counters = {
    "tp": 0, "tn": 0, "fp": 0, "fn": 0, "er": 0,
    "rejection_real": 0, "rejection_fake": 0
}

systemPrompt = getSystemPrompt(PromptITA)
def analyze_image(img_path, lab):
    try:
        messages = [
          {
              "role": "system",
              "content": [
                  {
                      "type": "text",
                      "text": systemPrompt
                  }
              ]
          },
          {"role": "user", "content": [{"type": "image", "image": str(img_path)}, {"type": "text", "text": PROMPT}]}
      ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        generated_ids = model.generate(**inputs, max_new_tokens=128)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

        print(f"\nImage: {img_path.name}")
        print("Raw Output:", output_text)
        # Metodo per fare il parsing usando l'output JSON
        text_raw = output_text[0].strip()
        # Clean potential markdown fences
        text_clean = re.sub(r"^```(?:json)?\s*([\s\S]*?)\s*```$", r"\1", text_raw.strip(), flags=re.MULTILINE)
        try:
            parsed = json.loads(text_clean)
            result = parsed.get("result")

            # Gestione di valori tipo lista o altro
            if isinstance(result, list) and result:
                result = result[0]
            prediction = str(result).strip().lower()
        except Exception as e:
            counters["er"] += 1
            print(f" JSON Parsing error: {e}")
            return
        # parsing se non si usa il JSON, non è molto efficace
        # match = re.search(r"(?:\[)?(yes|no|uncertain)(?:\])?", text.lower())
        # if not match:
        #     counters["er"] += 1
        #     print(" Parsing error, no match.")
        #     return
        #
        # prediction = match.group(1)

        if SHOW_IMAGES:
            Image.open(img_path).show()

        if lab == 1:  # Real
            if (prediction == "real face" or prediction == "real" or prediction == "[real face]" or
                    prediction == "agreed" or prediction == "[real]"  or prediction == "[no]" or prediction == "no"):
                counters["tn"] += 1
                print(" TN (real correctly identified)")
            elif prediction == "generated" or prediction == "[generated]" or prediction == "didn't agree" or prediction == "generated face" or prediction == "[generated face]" or prediction == "yes" or prediction == "[yes]":
                counters["fp"] += 1
                print(" FP (real misclassified as fake)")
            else:  # uncertain
                counters["rejection_real"] += 1
                print(" Rejection on real image")
        else:  # Fake
            if prediction == "generated" or prediction == "[generated]" or prediction == "didn't agree" or prediction == "generated face" or prediction == "[generated face]" or prediction == "yes" or prediction == "[yes]":
                counters["tp"] += 1
                print(" TP (fake correctly identified)")  # diamo importanza all'identificare il falso adesso
            elif (prediction == "real face" or prediction == "real" or prediction == "[real face]" or
                    prediction == "agreed" or prediction == "[real]" or prediction == "[no]" or prediction == "no"):
                counters["fn"] += 1
                print(" FN (fake misclassified as real)")
            else:  # uncertain
                counters["rejection_fake"] += 1
                print(" Rejection on fake image")

    except Exception as e:
        print(f" Error on {img_path}: {e}")
        counters["er"] += 1


# Main analysis loop
for img_path, label in tqdm(images_with_labels, desc=" Analyzing images"):
    analyze_image(img_path, label)

# Metrics
total_classified = counters["tp"] + counters["tn"] + counters["fp"] + counters["fn"]
accuracy = (counters["tp"] + counters["tn"]) / total_classified if total_classified else 0
precision = counters["tp"] / (counters["tp"] + counters["fp"]) if (counters["tp"] + counters["fp"]) else 0
recall = counters["tp"] / (counters["tp"] + counters["fn"]) if (counters["tp"] + counters["fn"]) else 0

total_real = counters["tp"] + counters["fn"] + counters["rejection_real"]
total_fake = counters["tn"] + counters["fp"] + counters["rejection_fake"]

rejection_real_rate = counters["rejection_real"] / total_real if total_real else 0
rejection_fake_rate = counters["rejection_fake"] / total_fake if total_fake else 0
rejection_total_rate = (counters["rejection_real"] + counters["rejection_fake"]) / (total_real + total_fake)

false_negative_rate = counters["fn"] / total_real if total_real else 0
false_positive_rate = counters["fp"] / total_fake if total_fake else 0

# Results
print("\n====== FINAL REPORT ======")
print(f"Total processed: {len(images_with_labels)}")
print(f"TP: {counters['tp']} | TN: {counters['tn']} | FP: {counters['fp']} | FN: {counters['fn']}")
print(f"Rejections on real: {counters['rejection_real']} | Rejections on fake: {counters['rejection_fake']}")
print(f"Text parsing errors: {counters['er']} ({(counters['er'] / len(images_with_labels)) * 100:.2f}%)\n")

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"False Negative Rate (real->fake): {false_negative_rate * 100:.2f}%")
print(f"False Positive Rate (fake->real): {false_positive_rate * 100:.2f}%")
print(f"Rejection Rate on real images: {rejection_real_rate * 100:.2f}%")
print(f"Rejection Rate on fake images: {rejection_fake_rate * 100:.2f}%")

# salvataggio in json
results = {
    "total_processed": len(images_with_labels),
    "total_real": len(real_images),
    "total_fake": len(fake_images),
    "TP": counters["tp"],
    "TN": counters["tn"],
    "FP": counters["fp"],
    "FN": counters["fn"],
    "rejection_real": counters["rejection_real"],
    "rejection_fake": counters["rejection_fake"],
    "text_parsing_errors": counters["er"],
    "text_parsing_error_rate": (counters["er"] / len(images_with_labels)) if len(images_with_labels) else 0,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "false_negative_rate": false_negative_rate,
    "false_positive_rate": false_positive_rate,
    "rejection_real_rate": rejection_real_rate,
    "rejection_fake_rate": rejection_fake_rate,
    "rejection_total_rate": rejection_total_rate,
    "system_prompt": systemPrompt,
    "user_prompt": PROMPT,
    "image_boost": True,
    "image_boost_value_min": min_pixels,
    "image_boost_value_max": max_pixels
}

# Crea cartella se non esiste
Path("resultsJSON").mkdir(exist_ok=True)

# Imposta lingua
language_tag = "ITA" if PromptITA else "ENG"

# Timestamp per identificare diversi tentativi
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Pulisci MODEL_NAME da caratteri non ammessi nei nomi file
safe_model_name = MODEL_NAME.replace(":", "_").replace("/", "_")

# Costruisci filename
filename = f"resultsJSON/real-vs-fake_{safe_model_name}_PromptType-{IndexPrompt}_{language_tag}_{timestamp}_result.json"

# Salva JSON
with open(filename, "w") as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {filename}.")

You choose this: I believe this person doesn’t exist. Based on the image, does it appear to be a real face or a generated one?



 Analyzing images:   1%|          | 1/100 [00:08<13:57,  8.46s/it]


Image: 52876.jpg
Raw Output: ['```json\n{\n  "result": "[generated]",\n  "explanation": "The image shows a highly detailed and realistic facial appearance with clear skin texture, precise eye makeup, and a well-defined hairstyle. The lighting and shadows also suggest a high level of realism. These characteristics are typically associated with generated images rather than real photographs."\n}\n```']
 FP (real misclassified as fake)


 Analyzing images:   2%|▏         | 2/100 [00:17<14:00,  8.57s/it]


Image: 59454.jpg
Raw Output: ['```json\n{\n  "result": "[generated]",\n  "explanation": "The skin texture appears overly smooth and uniform, which is a common characteristic of generated images. Additionally, the lighting and shadows do not match natural human skin tones, suggesting artificial enhancement. The overall appearance lacks the subtle variations and imperfections found in real human faces."\n}\n```']
 FP (real misclassified as fake)


...


Image: 4LG6SU9Z6D.jpg
Raw Output: ['```json\n{\n  "result": "[generated]",\n  "explanation": "The image shows a high level of detail and symmetry, which is unusual for a real human face. The lighting and shadows also seem too uniform and perfect for a natural human face. Additionally, the background and overall composition suggest that this is a digitally created image rather than a photograph of a real person."\n}\n```']
 TP (fake correctly identified)


 Analyzing images:  99%|█████████▉| 99/100 [14:40<00:08,  8.80s/it]


Image: 4W7S2GXAOF.jpg
Raw Output: ['```json\n{\n  "result": "[generated]",\n  "explanation": "The image shows a highly detailed and realistic depiction of a human face with smooth skin texture, symmetrical features, and natural lighting. The high level of detail and the absence of any imperfections suggest that this is a generated image rather than a real photograph."\n}\n```']
 TP (fake correctly identified)


 Analyzing images: 100%|██████████| 100/100 [14:50<00:00,  8.91s/it]


Image: 8IJBIRZMN5.jpg
Raw Output: ['```json\n{\n  "result": "[generated]",\n  "explanation": "The image shows a person with exaggerated features such as large eyes, a wide smile, and a prominent nose. These characteristics are often seen in generated images where digital tools are used to enhance facial features for artistic or aesthetic purposes. Additionally, the lighting and shadows on the face do not align with natural lighting conditions, further suggesting that this is a generated image."\n}\n```']
 TP (fake correctly identified)

====== FINAL REPORT ======
Total processed: 100
TP: 48 | TN: 2 | FP: 48 | FN: 2
Rejections on real: 0 | Rejections on fake: 0
Text parsing errors: 0 (0.00%)

Accuracy: 0.5000
Precision: 0.5000
Recall: 0.9600
False Negative Rate (real->fake): 4.00%
False Positive Rate (fake->real): 96.00%
Rejection Rate on real images: 0.00%
Rejection Rate on fake images: 0.00%
Results saved to resultsJSON/real-vs-fake_qwen2.5_PromptType-6_ENG_20250802-122119_result.jso